# Qasper Intermediate Extraction + Rule/IR Answer Composer

Standalone diagnostic notebook to test whether Qwen 0.5B is bottlenecked by evidence selection, reasoning/synthesis, or answer wording.

In [1]:
# Kaggle setup. Re-run this cell if the runtime is reset.
%pip install -q "sentence-transformers==3.0.1" "rank-bm25>=0.2.2" "transformers==4.44.2" "accelerate>=0.33.0" "pyarrow>=15.0.0" "pandas==2.2.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.

## Configuration and Utilities

This notebook reuses the adaptive context builder from notebook 18 inline, then adds intermediate evidence extraction and deterministic answer composition diagnostics. Gold evidence and answers are used only for scoring.

In [2]:
from __future__ import annotations

import math
import random
import re
import string
from dataclasses import dataclass
from pathlib import Path
from statistics import mean
from typing import Any
from urllib.parse import quote

import numpy as np
import pandas as pd


SPLIT = "validation"
LIMIT = 50                 # fixed pilot size; raise to 100 after the first successful Kaggle run
RANDOM_SEED = 13
TOP_K = 10
DENSE_MODEL_NAME = "intfloat/e5-small-v2"
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 96

CACHE_BASE_CONDITION = "dense_rag"  # compatibility constants for reused notebook utilities
CACHE_CANDIDATE_TOP_K = 10
CACHE_V3_TOP_K_VALUES = [10, 30]
CACHE_MAX_READER_CHUNKS = 6
CACHE_MIN_RELEVANCE_SCORE = 0.05
CACHE_CONFIDENCE_THRESHOLD = 0.85
CACHE_MIN_CHANGE_RATIO = 0.05
CACHE_MAX_MEMORY_WORDS = 220

EVIDENCE_POOL_TOP_K_VALUES = [10, 30]
EVIDENCE_MAX_SENTENCES = 12
EVIDENCE_MIN_SENTENCES = 4
EVIDENCE_MAX_WORDS = 1600
EVIDENCE_MIN_TOP_SCORE = 0.08
EVIDENCE_MAX_PER_CHUNK = 3
RUN_BASELINES = True
BASELINE_CONDITIONS = ["dense_rag_top10", "hybrid_rerank_reorder_top10_direct"]


@dataclass(frozen=True)
class Section:
    name: str
    paragraphs: tuple[str, ...]


@dataclass(frozen=True)
class Paper:
    paper_id: str
    title: str
    abstract: str
    sections: tuple[Section, ...]


@dataclass(frozen=True)
class GoldAnswer:
    free_form_answer: str = ""
    extractive_spans: tuple[str, ...] = ()
    yes_no: bool | None = None
    unanswerable: bool = False

    def texts(self) -> list[str]:
        values = []
        if self.free_form_answer:
            values.append(self.free_form_answer)
        values.extend(span for span in self.extractive_spans if span)
        if self.yes_no is not None:
            values.append("yes" if self.yes_no else "no")
        if self.unanswerable and not values:
            values.append("unanswerable")
        return values


@dataclass(frozen=True)
class Evidence:
    text: str
    section: str = ""


@dataclass(frozen=True)
class QAExample:
    example_id: str
    paper_id: str
    question_id: str
    question: str
    paper: Paper
    answers: tuple[GoldAnswer, ...]
    evidence: tuple[Evidence, ...]

    def gold_texts(self) -> list[str]:
        texts = []
        for answer in self.answers:
            texts.extend(answer.texts())
        return texts

    def evidence_texts(self) -> list[str]:
        return [item.text for item in self.evidence if item.text]


@dataclass(frozen=True)
class Chunk:
    paper_id: str
    chunk_id: str
    section: str
    text: str
    title: str = ""
    start_token: int = 0
    end_token: int = 0


@dataclass(frozen=True)
class RetrievedContext:
    chunk: Chunk
    score: float
    rank: int
    source: str

    def as_dict(self) -> dict[str, Any]:
        return {
            "paper_id": self.chunk.paper_id,
            "chunk_id": self.chunk.chunk_id,
            "section": self.chunk.section,
            "rank": self.rank,
            "score": self.score,
            "source": self.source,
            "text": self.chunk.text,
        }


def qasper_parquet_url(split: str) -> str:
    revision = quote("refs/convert/parquet", safe="")
    return f"https://huggingface.co/datasets/allenai/qasper/resolve/{revision}/qasper/{split}/0000.parquet"


def as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    return [value]


def parse_yes_no(value: Any) -> bool | None:
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text == "yes":
        return True
    if text == "no":
        return False
    return None


def parse_evidence(raw_evidence: Any) -> list[Evidence]:
    evidence = []
    for item in as_list(raw_evidence):
        if isinstance(item, dict):
            text = str(item.get("text") or item.get("evidence") or "").strip()
            section = str(item.get("section") or item.get("section_name") or "")
        else:
            text, section = str(item).strip(), ""
        if text:
            evidence.append(Evidence(text=text, section=section))
    return evidence


def parse_answers_and_evidence(qa: dict[str, Any]) -> tuple[list[GoldAnswer], list[Evidence]]:
    answers, evidence = [], []
    raw_answers = qa.get("answers") or qa.get("answer") or []
    for annotation in as_list(raw_answers):
        if isinstance(annotation, dict) and "answer" in annotation:
            payloads = as_list(annotation.get("answer"))
            evidence.extend(parse_evidence(annotation.get("evidence")))
        else:
            payloads = [annotation]
        for payload in payloads:
            if isinstance(payload, dict):
                answers.append(GoldAnswer(
                    free_form_answer=str(payload.get("free_form_answer") or "").strip(),
                    extractive_spans=tuple(str(s).strip() for s in as_list(payload.get("extractive_spans")) if str(s).strip()),
                    yes_no=parse_yes_no(payload.get("yes_no")),
                    unanswerable=bool(payload.get("unanswerable", False)),
                ))
            elif str(payload).strip():
                answers.append(GoldAnswer(free_form_answer=str(payload).strip()))
    evidence.extend(parse_evidence(qa.get("evidence")))
    return answers, evidence


def iter_question_items(qas: Any):
    if isinstance(qas, list):
        for item in qas:
            if isinstance(item, dict):
                yield item
        return
    if not isinstance(qas, dict):
        return
    questions = as_list(qas.get("question"))
    for index, question in enumerate(questions):
        item = {"question": question}
        for field_name, field_values in qas.items():
            values = as_list(field_values)
            if index < len(values):
                item[field_name] = values[index]
        yield item


def parse_paper(row: dict[str, Any]) -> Paper:
    full_text = row.get("full_text") or {}
    section_names = as_list(full_text.get("section_name"))
    paragraphs_by_section = as_list(full_text.get("paragraphs"))
    sections = []
    for index, paragraphs in enumerate(paragraphs_by_section):
        name = str(section_names[index]) if index < len(section_names) else f"section_{index}"
        clean_paragraphs = tuple(str(p).strip() for p in as_list(paragraphs) if str(p).strip())
        if clean_paragraphs:
            sections.append(Section(name=name, paragraphs=clean_paragraphs))
    return Paper(
        paper_id=str(row.get("id") or row.get("paper_id") or ""),
        title=str(row.get("title") or ""),
        abstract=str(row.get("abstract") or ""),
        sections=tuple(sections),
    )


def normalize_qasper_row(row: dict[str, Any]) -> list[QAExample]:
    paper = parse_paper(row)
    examples = []
    for index, qa in enumerate(iter_question_items(row.get("qas") or {})):
        question = str(qa.get("question") or "").strip()
        if not question:
            continue
        question_id = str(qa.get("question_id") or f"{paper.paper_id}-{index}")
        answers, evidence = parse_answers_and_evidence(qa)
        examples.append(QAExample(
            example_id=f"{paper.paper_id}:{question_id}",
            paper_id=paper.paper_id,
            question_id=question_id,
            question=question,
            paper=paper,
            answers=tuple(answers),
            evidence=tuple(evidence),
        ))
    return examples


def load_qasper_examples(split: str, limit: int | None = None, local_parquet_path: str | None = None) -> list[QAExample]:
    frame = pd.read_parquet(local_parquet_path or qasper_parquet_url(split))
    examples = []
    for row in frame.to_dict(orient="records"):
        examples.extend(normalize_qasper_row(row))
        if limit is not None and len(examples) >= limit:
            return examples[:limit]
    return examples


def chunk_paper(paper: Paper, chunk_size_tokens: int = 160, chunk_overlap_tokens: int = 30) -> list[Chunk]:
    items = []
    if paper.title:
        items.append(("title", paper.title))
    if paper.abstract:
        items.append(("abstract", paper.abstract))
    for section in paper.sections:
        for paragraph_index, paragraph in enumerate(section.paragraphs):
            items.append((f"{section.name} / paragraph {paragraph_index + 1}", paragraph))
    chunks = []
    step = chunk_size_tokens - chunk_overlap_tokens
    for section_name, section_text in items:
        tokens = section_text.split()
        start = 0
        while start < len(tokens):
            end = min(start + chunk_size_tokens, len(tokens))
            text = " ".join(tokens[start:end]).strip()
            if text:
                chunks.append(Chunk(paper.paper_id, f"{paper.paper_id}:{len(chunks)}", section_name, text, paper.title, start, end))
            if end == len(tokens):
                break
            start += step
    return chunks


TOKEN_RE = re.compile(r"[A-Za-z0-9_]+")
ARTICLES_RE = re.compile(r"\b(a|an|the)\b", flags=re.IGNORECASE)
DENSE_ENCODERS = {}
HF_MODEL = None
HF_TOKENIZER = None
CROSS_ENCODERS = {}


def tokenize(text: str) -> list[str]:
    return [m.group(0).lower() for m in TOKEN_RE.finditer(text)]


def normalize_answer(text: str) -> str:
    text = str(text or "").lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = ARTICLES_RE.sub(" ", text)
    return " ".join(text.split())


def exact_match(prediction: str, gold_texts: list[str]) -> float:
    pred = normalize_answer(prediction)
    return float(any(pred == normalize_answer(gold) for gold in gold_texts if gold))


def token_f1(prediction: str, gold_texts: list[str]) -> float:
    pred_tokens = normalize_answer(prediction).split()
    if not pred_tokens:
        return 0.0
    best = 0.0
    for gold in gold_texts:
        gold_tokens = normalize_answer(gold).split()
        if not gold_tokens:
            continue
        common = set(pred_tokens) & set(gold_tokens)
        overlap = sum(min(pred_tokens.count(t), gold_tokens.count(t)) for t in common)
        if overlap == 0:
            continue
        precision = overlap / len(pred_tokens)
        recall = overlap / len(gold_tokens)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


def normalized_contains(text: str, needle: str) -> bool:
    text_norm, needle_norm = normalize_answer(text), normalize_answer(needle)
    return bool(needle_norm and needle_norm in text_norm)


def context_precision(contexts: list[str], gold_texts: list[str], evidence_texts: list[str]) -> float:
    if not contexts:
        return 0.0
    refs = [r for r in evidence_texts + gold_texts if r]
    if not refs:
        return 0.0
    return sum(any(normalized_contains(ctx, ref) for ref in refs) for ctx in contexts) / len(contexts)


def context_recall(contexts: list[str], gold_texts: list[str], evidence_texts: list[str]) -> float:
    refs = [r for r in evidence_texts + gold_texts if r]
    if not refs:
        return 0.0
    joined = "\n".join(contexts)
    return sum(normalized_contains(joined, ref) for ref in refs) / len(refs)


def faithfulness_heuristic(prediction: str, contexts: list[str]) -> float:
    pred_tokens = set(normalize_answer(prediction).split())
    context_tokens = set(normalize_answer("\n".join(contexts)).split())
    if not pred_tokens or not context_tokens:
        return 0.0
    return len(pred_tokens & context_tokens) / len(pred_tokens)


def answer_relevancy_heuristic(prediction: str, question: str, gold_texts: list[str]) -> float:
    pred_tokens = set(normalize_answer(prediction).split())
    ref_tokens = set(normalize_answer(question).split())
    for gold in gold_texts:
        ref_tokens.update(normalize_answer(gold).split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    return len(pred_tokens & ref_tokens) / len(pred_tokens)


def score_prediction(prediction: str, example: QAExample, contexts: list[str]) -> dict[str, float]:
    gold_texts = example.gold_texts()
    evidence_texts = example.evidence_texts()
    return {
        "exact_match": exact_match(prediction, gold_texts),
        "token_f1": token_f1(prediction, gold_texts),
        "context_precision": context_precision(contexts, gold_texts, evidence_texts),
        "context_recall": context_recall(contexts, gold_texts, evidence_texts),
        "faithfulness": faithfulness_heuristic(prediction, contexts),
        "answer_relevancy": answer_relevancy_heuristic(prediction, example.question, gold_texts),
    }


def rank_chunks(chunks: list[Chunk], scores: list[float], top_k: int, source: str) -> list[RetrievedContext]:
    ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)
    return [RetrievedContext(chunks[i], float(score), rank, source) for rank, (i, score) in enumerate(ranked[:top_k], start=1)]


def bm25_scores(corpus_tokens: list[list[str]], query_tokens: list[str]) -> list[float]:
    try:
        from rank_bm25 import BM25Okapi
        return BM25Okapi(corpus_tokens).get_scores(query_tokens).tolist()
    except Exception:
        pass
    doc_count = len(corpus_tokens)
    avgdl = sum(len(t) for t in corpus_tokens) / max(1, doc_count)
    dfs = {}
    for tokens in corpus_tokens:
        for token in set(tokens):
            dfs[token] = dfs.get(token, 0) + 1
    scores = []
    for tokens in corpus_tokens:
        score = 0.0
        doc_len = len(tokens) or 1
        for token in query_tokens:
            freq = tokens.count(token)
            if freq == 0:
                continue
            df = dfs.get(token, 0)
            idf = math.log(1 + (doc_count - df + 0.5) / (df + 0.5))
            denom = freq + 1.5 * (1 - 0.75 + 0.75 * doc_len / avgdl)
            score += idf * (freq * 2.5) / denom
        scores.append(score)
    return scores


def retrieve_bm25(question: str, chunks: list[Chunk], top_k: int) -> list[RetrievedContext]:
    return rank_chunks(chunks, bm25_scores([tokenize(c.text) for c in chunks], tokenize(question)), top_k, "bm25")


def retrieve_dense(question: str, chunks: list[Chunk], top_k: int, model_name: str = DENSE_MODEL_NAME) -> list[RetrievedContext]:
    from sentence_transformers import SentenceTransformer
    encoder = DENSE_ENCODERS.get(model_name)
    if encoder is None:
        encoder = SentenceTransformer(model_name)
        DENSE_ENCODERS[model_name] = encoder
    query = encoder.encode([f"query: {question}"], normalize_embeddings=True)[0]
    passages = encoder.encode([f"passage: {c.text}" for c in chunks], normalize_embeddings=True)
    scores = np.asarray(passages) @ np.asarray(query)
    return rank_chunks(chunks, scores.tolist(), top_k, "dense")


def retrieve_hybrid_rrf(question: str, chunks: list[Chunk], top_k: int, model_name: str = DENSE_MODEL_NAME, rrf_k: int = 60) -> list[RetrievedContext]:
    pool_k = max(top_k, min(len(chunks), top_k * 4))
    bm25_results = retrieve_bm25(question, chunks, pool_k)
    dense_results = retrieve_dense(question, chunks, pool_k, model_name=model_name)
    scores = {}
    for result_list in [bm25_results, dense_results]:
        for result in result_list:
            scores[result.chunk.chunk_id] = scores.get(result.chunk.chunk_id, 0.0) + 1.0 / (rrf_k + result.rank)
    chunk_by_id = {chunk.chunk_id: chunk for chunk in chunks}
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [RetrievedContext(chunk_by_id[chunk_id], float(score), rank, "hybrid_rrf") for rank, (chunk_id, score) in enumerate(ranked[:top_k], start=1)]


def rerank_with_cross_encoder(question: str, contexts: list[RetrievedContext], model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2") -> list[RetrievedContext]:
    from sentence_transformers import CrossEncoder
    cross_encoder = CROSS_ENCODERS.get(model_name)
    if cross_encoder is None:
        cross_encoder = CrossEncoder(model_name)
        CROSS_ENCODERS[model_name] = cross_encoder
    pairs = [(question, context.chunk.text) for context in contexts]
    scores = cross_encoder.predict(pairs).tolist()
    ranked = sorted(zip(contexts, scores), key=lambda item: item[1], reverse=True)
    return [
        RetrievedContext(context.chunk, float(score), rank, "hybrid_rerank")
        for rank, (context, score) in enumerate(ranked, start=1)
    ]


def u_shape_reorder(contexts: list[RetrievedContext]) -> list[RetrievedContext]:
    reordered = [None] * len(contexts)
    left, right = 0, len(contexts) - 1
    place_left = True
    for context in contexts:
        if place_left:
            reordered[left] = context
            left += 1
        else:
            reordered[right] = context
            right -= 1
        place_left = not place_left
    return [
        RetrievedContext(context.chunk, context.score, rank, "hybrid_rerank_reorder")
        for rank, context in enumerate(reordered, start=1)
        if context is not None
    ]


def retrieve_hybrid_rerank_reorder(question: str, chunks: list[Chunk], top_k: int) -> list[RetrievedContext]:
    pool_k = max(top_k, min(len(chunks), top_k * 4))
    candidates = retrieve_hybrid_rrf(question, chunks, pool_k)
    reranked = rerank_with_cross_encoder(question, candidates)
    return u_shape_reorder(reranked[:top_k])


def retrieve(condition: str, question: str, chunks: list[Chunk], top_k: int) -> list[RetrievedContext]:
    if condition == "bm25_rag":
        return retrieve_bm25(question, chunks, top_k)
    if condition == "dense_rag":
        return retrieve_dense(question, chunks, top_k)
    if condition == "hybrid_rag":
        return retrieve_hybrid_rrf(question, chunks, top_k)
    if condition == "hybrid_rerank_reorder":
        return retrieve_hybrid_rerank_reorder(question, chunks, top_k)
    raise ValueError(f"Unsupported retrieval condition: {condition}")


def format_contexts(contexts: list[RetrievedContext]) -> str:
    if not contexts:
        return "[No context provided]"
    return "\n\n".join(
        f"[{ctx.rank}] section={ctx.chunk.section} score={ctx.score:.4f}\n{ctx.chunk.text}"
        for ctx in contexts
    )


def load_generator():
    global HF_MODEL, HF_TOKENIZER
    if HF_MODEL is not None and HF_TOKENIZER is not None:
        return HF_TOKENIZER, HF_MODEL
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    HF_TOKENIZER = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME, trust_remote_code=True)
    HF_MODEL = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, torch_dtype="auto", trust_remote_code=True)
    HF_MODEL = HF_MODEL.to("cuda" if torch.cuda.is_available() else "cpu")
    HF_MODEL.eval()
    return HF_TOKENIZER, HF_MODEL


def generate_text(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    import torch
    tokenizer, model = load_generator()
    if getattr(tokenizer, "chat_template", None):
        rendered = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    else:
        rendered = prompt
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=30000)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def build_direct_prompt(question: str, contexts: list[RetrievedContext]) -> str:
    return (
        "Answer the question using only the provided context. If the answer is not supported by the context, say 'unanswerable'.\n\n"
        f"Context:\n{format_contexts(contexts)}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def lexical_overlap(question: str, text: str) -> float:
    q_tokens, t_tokens = set(normalize_answer(question).split()), set(normalize_answer(text).split())
    if not q_tokens:
        return 0.0
    return len(q_tokens & t_tokens) / len(q_tokens)


def select_reader_contexts(question: str, contexts: list[RetrievedContext]) -> list[RetrievedContext]:
    rescored = []
    for context in contexts:
        relevance = 0.65 * (1.0 / max(context.rank, 1)) + 0.35 * lexical_overlap(question, context.chunk.text)
        rescored.append(RetrievedContext(context.chunk, float(relevance), context.rank, context.source))
    filtered = [ctx for ctx in rescored if ctx.score >= CACHE_MIN_RELEVANCE_SCORE] or rescored
    return sorted(filtered, key=lambda ctx: ctx.score, reverse=True)[:CACHE_MAX_READER_CHUNKS]


def build_cache_key(example: QAExample, retriever_condition: str, top_k: int) -> str:
    return f"{example.question_id}|{example.paper_id}|{retriever_condition}|top_k={top_k}"


def build_memory_update_prompt(question: str, memory: str, context: RetrievedContext) -> str:
    current_memory = memory.strip() or "[empty]"
    return (
        "Cache memory update task.\n"
        "Read one context chunk and update the memory only with facts that help answer the question. "
        "Do not answer the question yet. If the chunk adds nothing useful, keep the memory unchanged.\n\n"
        f"Question:\n{question}\n\n"
        f"Current memory:\n{current_memory}\n\n"
        f"New chunk [{context.rank}] section={context.chunk.section}:\n{context.chunk.text}\n\n"
        "Return exactly:\n"
        "MEMORY: <short updated memory>\n"
        "CONFIDENCE: <number from 0 to 1 showing whether memory is sufficient to answer>"
    )


def parse_memory_update(text: str) -> tuple[str, float]:
    memory = ""
    confidence = 0.0
    memory_match = re.search(r"MEMORY\s*:\s*(.*?)(?:\n\s*CONFIDENCE\s*:|$)", text or "", flags=re.I | re.S)
    if memory_match:
        memory = memory_match.group(1).strip()
    confidence_match = re.search(r"CONFIDENCE\s*:\s*([01](?:\.\d+)?)", text or "", flags=re.I)
    if confidence_match:
        confidence = min(1.0, max(0.0, float(confidence_match.group(1))))
    return memory, confidence


def split_sentences(text: str) -> list[str]:
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text or "") if part.strip()]


def trim_words(text: str, max_words: int) -> str:
    words = text.split()
    return " ".join(words[:max_words]).strip() if len(words) > max_words else text.strip()


def fallback_memory_update(question: str, memory: str, context_text: str) -> str:
    selected = [sentence for sentence in split_sentences(context_text) if lexical_overlap(question, sentence) > 0.0]
    if not selected:
        return memory
    merged = " ".join(part for part in [memory.strip(), " ".join(selected)] if part)
    seen, kept = set(), []
    for sentence in split_sentences(merged):
        key = normalize_answer(sentence)
        if key and key not in seen:
            seen.add(key)
            kept.append(sentence)
    return trim_words(" ".join(kept), CACHE_MAX_MEMORY_WORDS)


def memory_change_ratio(previous: str, current: str) -> float:
    prev_tokens = set(normalize_answer(previous).split())
    cur_tokens = set(normalize_answer(current).split())
    if not cur_tokens:
        return 0.0
    if not prev_tokens:
        return 1.0
    return len(cur_tokens - prev_tokens) / len(cur_tokens)


def build_cache_memory(example: QAExample, chunks: list[Chunk]) -> dict[str, Any]:
    candidates = retrieve(CACHE_BASE_CONDITION, example.question, chunks, CACHE_CANDIDATE_TOP_K)
    reader_contexts = select_reader_contexts(example.question, candidates)
    memory = ""
    confidence = 0.0
    steps = []
    stopped_reason = "exhausted_candidates"
    for context in reader_contexts:
        previous = memory
        raw = generate_text(build_memory_update_prompt(example.question, memory, context), max_new_tokens=160)
        parsed_memory, parsed_confidence = parse_memory_update(raw)
        if not parsed_memory:
            parsed_memory = fallback_memory_update(example.question, memory, context.chunk.text)
        memory = trim_words(parsed_memory, CACHE_MAX_MEMORY_WORDS)
        confidence = max(confidence, parsed_confidence)
        if memory and confidence == 0.0:
            confidence = min(0.8, 0.25 + 0.1 * len(steps))
        change_ratio = memory_change_ratio(previous, memory)
        steps.append({
            "chunk_id": context.chunk.chunk_id,
            "input_rank": context.rank,
            "relevance_score": context.score,
            "confidence": confidence,
            "memory_changed": normalize_answer(previous) != normalize_answer(memory),
            "memory_word_count": len(memory.split()),
            "raw_update": raw[:500],
        })
        if confidence >= CACHE_CONFIDENCE_THRESHOLD:
            stopped_reason = "confidence_threshold"
            break
        if previous and change_ratio < CACHE_MIN_CHANGE_RATIO:
            stopped_reason = "low_memory_change"
            break
    return {
        "cache_key": build_cache_key(example, CACHE_BASE_CONDITION, CACHE_CANDIDATE_TOP_K),
        "candidate_contexts": candidates,
        "reader_contexts": reader_contexts,
        "memory": memory,
        "confidence": confidence,
        "steps": steps,
        "stopped_reason": stopped_reason,
    }


def build_final_answer_prompt(question: str, memory: str) -> str:
    memory_block = memory.strip() or "[No useful memory was extracted]"
    return (
        "Answer the question using only the condensed memory. If the memory is insufficient, say 'unanswerable'.\n\n"
        f"Condensed memory:\n{memory_block}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def memory_context(example: QAExample, memory: str, confidence: float, cache_key: str) -> list[RetrievedContext]:
    if not memory.strip():
        return []
    chunk = Chunk(
        paper_id=example.paper_id,
        chunk_id=f"{example.paper_id}:cache_memory:{example.question_id}",
        section="cache_memory",
        text=memory.strip(),
        title=example.paper.title,
        start_token=0,
        end_token=len(memory.split()),
    )
    return [RetrievedContext(chunk, confidence, 1, f"cache_memory:{cache_key}")]


def fact_key(text: str) -> str:
    return normalize_answer(text)[:220]


def build_memory_update_prompt_v2(question: str, facts: list[dict[str, Any]], context: RetrievedContext) -> str:
    current_facts = facts_to_memory(facts) if facts else "[empty]"
    return (
        "Evidence-guarded cache memory update task.\n"
        "Read one context chunk and extract only facts that help answer the question. "
        "Do not compress facts into one sentence. Keep source evidence attached to every fact.\n\n"
        f"Question:\n{question}\n\n"
        f"Current facts:\n{current_facts}\n\n"
        f"New chunk source_rank={context.rank} section={context.chunk.section}:\n{context.chunk.text}\n\n"
        "Return exactly this schema:\n"
        "FACTS:\n"
        "- source_rank=<rank>; quote=\"<short exact span from the chunk>\"; fact=<short useful fact>\n"
        "CONFIDENCE: <number from 0 to 1 showing whether the retained facts are sufficient>\n"
        "If the chunk adds no useful fact, return FACTS: with no bullet and CONFIDENCE: 0."
    )


def parse_confidence(text: str) -> float:
    confidence_match = re.search(r"CONFIDENCE\s*:\s*([01](?:\.\d+)?)", text or "", flags=re.I)
    if not confidence_match:
        return 0.0
    return min(1.0, max(0.0, float(confidence_match.group(1))))


def parse_memory_update_v2(text: str, context: RetrievedContext) -> tuple[list[dict[str, Any]], float]:
    facts = []
    for line in (text or "").splitlines():
        line = line.strip().lstrip("-").strip()
        if not line or "fact" not in line.lower():
            continue
        rank_match = re.search(r"source_rank\s*=\s*(\d+)", line, flags=re.I)
        quote_match = re.search(r"quote\s*=\s*['\"]([^'\"]+)['\"]", line, flags=re.I)
        fact_match = re.search(r"fact\s*=\s*(.+)$", line, flags=re.I)
        if not fact_match:
            continue
        fact_text = fact_match.group(1).strip(" ;")
        quote_text = quote_match.group(1).strip() if quote_match else ""
        source_rank = int(rank_match.group(1)) if rank_match else context.rank
        if fact_text:
            facts.append({
                "fact": trim_words(fact_text, 45),
                "quote": trim_words(quote_text, 40),
                "source_rank": source_rank,
                "chunk_id": context.chunk.chunk_id,
                "section": context.chunk.section,
                "relevance_score": context.score,
            })
    return facts, parse_confidence(text)


def fallback_facts_v2(question: str, context: RetrievedContext, max_facts: int = 2) -> list[dict[str, Any]]:
    selected = []
    for sentence in split_sentences(context.chunk.text):
        if lexical_overlap(question, sentence) > 0.0:
            selected.append(sentence)
    if not selected:
        return []
    return [
        {
            "fact": trim_words(sentence, 45),
            "quote": trim_words(sentence, 40),
            "source_rank": context.rank,
            "chunk_id": context.chunk.chunk_id,
            "section": context.chunk.section,
            "relevance_score": context.score,
        }
        for sentence in selected[:max_facts]
    ]


def merge_facts(existing: list[dict[str, Any]], new_facts: list[dict[str, Any]], max_facts: int = 8) -> list[dict[str, Any]]:
    by_key = {}
    for fact in existing + new_facts:
        key = fact_key(fact.get("fact", ""))
        if not key:
            continue
        previous = by_key.get(key)
        if previous is None:
            by_key[key] = fact
            continue
        previous_rank = int(previous.get("source_rank", 999))
        fact_rank = int(fact.get("source_rank", 999))
        previous_score = float(previous.get("relevance_score", 0.0))
        fact_score = float(fact.get("relevance_score", 0.0))
        if (fact_score, -fact_rank) > (previous_score, -previous_rank):
            by_key[key] = fact
    ranked = sorted(
        by_key.values(),
        key=lambda fact: (-float(fact.get("relevance_score", 0.0)), int(fact.get("source_rank", 999)), fact_key(fact.get("fact", ""))),
    )
    return ranked[:max_facts]


def facts_to_memory(facts: list[dict[str, Any]]) -> str:
    lines = []
    for fact in facts:
        quote = fact.get("quote") or ""
        quote_suffix = f" (quote: \"{quote}\")" if quote else ""
        lines.append(f"- [source {fact.get('source_rank', '?')}] {fact.get('fact', '').strip()}{quote_suffix}")
    return "\n".join(line for line in lines if line.strip())


def guard_confidence(raw_confidence: float, facts: list[dict[str, Any]], memory: str) -> float:
    confidence = min(1.0, max(0.0, raw_confidence))
    if len(facts) < 2:
        confidence = min(confidence, 0.55)
    if len(memory.split()) < 35:
        confidence = min(confidence, 0.65)
    if not any(fact.get("quote") for fact in facts):
        confidence = min(confidence, 0.70)
    return confidence


def select_evidence_snippets(facts: list[dict[str, Any]], candidates: list[RetrievedContext], max_snippets: int = 3) -> list[RetrievedContext]:
    by_chunk_id = {context.chunk.chunk_id: context for context in candidates}
    selected, seen = [], set()
    for fact in facts:
        chunk_id = fact.get("chunk_id")
        if chunk_id in by_chunk_id and chunk_id not in seen:
            selected.append(by_chunk_id[chunk_id])
            seen.add(chunk_id)
        if len(selected) >= max_snippets:
            return selected
    for context in candidates:
        if context.chunk.chunk_id not in seen:
            selected.append(context)
            seen.add(context.chunk.chunk_id)
        if len(selected) >= max_snippets:
            break
    return [
        RetrievedContext(context.chunk, context.score, rank, "cache_memory_v2_evidence")
        for rank, context in enumerate(selected, start=2)
    ]


def memory_context_v2(example: QAExample, memory: str, confidence: float, cache_key: str) -> list[RetrievedContext]:
    if not memory.strip():
        return []
    chunk = Chunk(
        paper_id=example.paper_id,
        chunk_id=f"{example.paper_id}:cache_memory_v2:{example.question_id}",
        section="cache_memory_v2",
        text=memory.strip(),
        title=example.paper.title,
        start_token=0,
        end_token=len(memory.split()),
    )
    return [RetrievedContext(chunk, confidence, 1, f"cache_memory_v2:{cache_key}")]


def build_final_answer_prompt_v2(question: str, memory: str, evidence_contexts: list[RetrievedContext]) -> str:
    memory_block = memory.strip() or "[No useful memory was extracted]"
    return (
        "Answer the question using only the condensed memory and evidence snippets. "
        "If the answer is not supported, say 'unanswerable'.\n\n"
        f"Condensed memory:\n{memory_block}\n\n"
        f"Evidence snippets:\n{format_contexts(evidence_contexts)}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def build_final_answer_prompt_v3(question: str, memory: str, contexts: list[RetrievedContext]) -> str:
    memory_block = memory.strip() or "[No useful memory guide was extracted]"
    return (
        "Answer the question using the retrieved chunks as the source of truth. "
        "The memory guide is only a reading aid; verify every useful fact against the retrieved chunks. "
        "If the chunks do not support an answer, say 'unanswerable'.\n\n"
        f"Memory guide:\n{memory_block}\n\n"
        f"Retrieved chunks:\n{format_contexts(contexts)}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def build_cache_memory_v2(example: QAExample, chunks: list[Chunk], candidate_top_k: int = CACHE_CANDIDATE_TOP_K) -> dict[str, Any]:
    candidates = retrieve(CACHE_BASE_CONDITION, example.question, chunks, candidate_top_k)
    reader_contexts = select_reader_contexts(example.question, candidates)
    raw_retrieval_context_recall = context_recall(
        [context.chunk.text for context in candidates],
        example.gold_texts(),
        example.evidence_texts(),
    )
    facts: list[dict[str, Any]] = []
    memory = ""
    confidence_raw = 0.0
    confidence_guarded = 0.0
    steps = []
    stopped_reason = "exhausted_candidates"
    for context in reader_contexts:
        previous_memory = memory
        previous_fact_count = len(facts)
        raw = generate_text(build_memory_update_prompt_v2(example.question, facts, context), max_new_tokens=220)
        parsed_facts, parsed_confidence = parse_memory_update_v2(raw, context)
        if not parsed_facts:
            parsed_facts = fallback_facts_v2(example.question, context)
        facts = merge_facts(facts, parsed_facts, max_facts=8)
        memory = trim_words(facts_to_memory(facts), CACHE_MAX_MEMORY_WORDS)
        confidence_raw = max(confidence_raw, parsed_confidence)
        if facts and confidence_raw == 0.0:
            confidence_raw = min(0.75, 0.25 + 0.08 * len(facts))
        confidence_guarded = guard_confidence(confidence_raw, facts, memory)
        change_ratio = memory_change_ratio(previous_memory, memory)
        steps.append({
            "chunk_id": context.chunk.chunk_id,
            "input_rank": context.rank,
            "relevance_score": context.score,
            "fact_count": len(facts),
            "new_fact_count": max(0, len(facts) - previous_fact_count),
            "memory_changed": normalize_answer(previous_memory) != normalize_answer(memory),
            "memory_word_count": len(memory.split()),
            "confidence_raw": confidence_raw,
            "confidence_guarded": confidence_guarded,
            "raw_update": raw[:500],
        })
        if confidence_guarded >= CACHE_CONFIDENCE_THRESHOLD and len(facts) >= 2 and len(memory.split()) >= 35:
            stopped_reason = "confidence_threshold_guarded"
            break
        if previous_memory and previous_fact_count == len(facts) and change_ratio < CACHE_MIN_CHANGE_RATIO:
            stopped_reason = "low_memory_change"
            break
    evidence_contexts = select_evidence_snippets(facts, candidates, max_snippets=3)
    final_contexts = ([memory] if memory.strip() else []) + [context.chunk.text for context in evidence_contexts]
    final_context_recall = context_recall(final_contexts, example.gold_texts(), example.evidence_texts())
    return {
        "cache_key": build_cache_key(example, CACHE_BASE_CONDITION, candidate_top_k),
        "candidate_top_k": candidate_top_k,
        "candidate_contexts": candidates,
        "reader_contexts": reader_contexts,
        "facts": facts,
        "memory": memory,
        "evidence_contexts": evidence_contexts,
        "confidence_raw": confidence_raw,
        "confidence_guarded": confidence_guarded,
        "raw_retrieval_context_recall": raw_retrieval_context_recall,
        "final_context_recall": final_context_recall,
        "steps": steps,
        "stopped_reason": stopped_reason,
    }


def run_baseline_condition(example: QAExample, condition: str) -> dict[str, Any]:
    chunks = chunk_paper(example.paper)
    if condition == "dense_rag_top10":
        contexts = retrieve_dense(example.question, chunks, TOP_K)
        prompt = build_direct_prompt(example.question, contexts)
    elif condition == "hybrid_rerank_reorder_top10_direct":
        contexts = retrieve_hybrid_rerank_reorder(example.question, chunks, TOP_K)
        prompt = build_direct_prompt(example.question, contexts)
    else:
        raise ValueError(condition)
    prediction = generate_text(prompt)
    metrics = score_prediction(prediction, example, [ctx.chunk.text for ctx in contexts])
    return {
        "example_id": example.example_id,
        "condition": condition,
        "prediction": prediction,
        "metrics": metrics,
        "retrieved_contexts": [ctx.as_dict() for ctx in contexts],
    }


def run_cache_memory_condition(example: QAExample) -> dict[str, Any]:
    chunks = chunk_paper(example.paper)
    cache = build_cache_memory(example, chunks)
    contexts = memory_context(example, cache["memory"], cache["confidence"], cache["cache_key"])
    prediction = generate_text(build_final_answer_prompt(example.question, cache["memory"]))
    metrics = score_prediction(prediction, example, [ctx.chunk.text for ctx in contexts])
    return {
        "example_id": example.example_id,
        "condition": "cache_memory_rag",
        "prediction": prediction,
        "metrics": metrics,
        "retrieved_contexts": [ctx.as_dict() for ctx in contexts],
        "candidate_contexts": [ctx.as_dict() for ctx in cache["candidate_contexts"]],
        "cache_memory_trace": {
            "cache_key": cache["cache_key"],
            "confidence": cache["confidence"],
            "stopped_reason": cache["stopped_reason"],
            "candidate_count": len(cache["candidate_contexts"]),
            "reader_context_count": len(cache["reader_contexts"]),
            "memory_word_count": len(cache["memory"].split()),
            "steps": cache["steps"],
        },
    }


def run_cache_memory_condition_v2(example: QAExample) -> dict[str, Any]:
    chunks = chunk_paper(example.paper)
    cache = build_cache_memory_v2(example, chunks)
    contexts = memory_context_v2(example, cache["memory"], cache["confidence_guarded"], cache["cache_key"]) + cache["evidence_contexts"]
    prediction = generate_text(build_final_answer_prompt_v2(example.question, cache["memory"], cache["evidence_contexts"]))
    metrics = score_prediction(prediction, example, [ctx.chunk.text for ctx in contexts])
    return {
        "example_id": example.example_id,
        "condition": "cache_memory_rag_v2",
        "prediction": prediction,
        "metrics": metrics,
        "retrieved_contexts": [ctx.as_dict() for ctx in contexts],
        "candidate_contexts": [ctx.as_dict() for ctx in cache["candidate_contexts"]],
        "cache_memory_trace": {
            "cache_key": cache["cache_key"],
            "confidence": cache["confidence_guarded"],
            "confidence_raw": cache["confidence_raw"],
            "confidence_guarded": cache["confidence_guarded"],
            "stopped_reason": cache["stopped_reason"],
            "candidate_count": len(cache["candidate_contexts"]),
            "reader_context_count": len(cache["reader_contexts"]),
            "fact_count": len(cache["facts"]),
            "memory_word_count": len(cache["memory"].split()),
            "raw_retrieval_context_recall": cache["raw_retrieval_context_recall"],
            "final_context_recall": cache["final_context_recall"],
            "facts": cache["facts"],
            "steps": cache["steps"],
        },
    }


def run_cache_memory_condition_v3(example: QAExample, candidate_top_k: int) -> dict[str, Any]:
    chunks = chunk_paper(example.paper)
    cache = build_cache_memory_v2(example, chunks, candidate_top_k=candidate_top_k)
    memory_contexts = memory_context_v2(example, cache["memory"], cache["confidence_guarded"], cache["cache_key"])
    contexts = memory_contexts + cache["candidate_contexts"]
    prediction = generate_text(build_final_answer_prompt_v3(example.question, cache["memory"], cache["candidate_contexts"]))
    metrics = score_prediction(prediction, example, [ctx.chunk.text for ctx in contexts])
    final_contexts = ([cache["memory"]] if cache["memory"].strip() else []) + [ctx.chunk.text for ctx in cache["candidate_contexts"]]
    return {
        "example_id": example.example_id,
        "condition": f"cache_memory_rag_v3_top{candidate_top_k}",
        "prediction": prediction,
        "metrics": metrics,
        "retrieved_contexts": [ctx.as_dict() for ctx in contexts],
        "candidate_contexts": [ctx.as_dict() for ctx in cache["candidate_contexts"]],
        "cache_memory_trace": {
            "cache_key": cache["cache_key"],
            "confidence": cache["confidence_guarded"],
            "confidence_raw": cache["confidence_raw"],
            "confidence_guarded": cache["confidence_guarded"],
            "stopped_reason": cache["stopped_reason"],
            "candidate_top_k": candidate_top_k,
            "candidate_count": len(cache["candidate_contexts"]),
            "reader_context_count": len(cache["reader_contexts"]),
            "fact_count": len(cache["facts"]),
            "memory_word_count": len(cache["memory"].split()),
            "raw_retrieval_context_recall": cache["raw_retrieval_context_recall"],
            "final_context_recall": context_recall(final_contexts, example.gold_texts(), example.evidence_texts()),
            "facts": cache["facts"],
            "steps": cache["steps"],
        },
    }


def summarize(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for condition in sorted({record["condition"] for record in records}):
        condition_records = [record for record in records if record["condition"] == condition]
        row = {"condition": condition, "count": float(len(condition_records))}
        for metric in ["token_f1", "exact_match", "context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
            row[metric] = mean(record["metrics"][metric] for record in condition_records)
        for diagnostic in [
            "raw_retrieval_context_recall",
            "final_context_recall",
            "candidate_top_k",
            "fact_count",
            "memory_word_count",
            "confidence_raw",
            "confidence_guarded",
        ]:
            values = [
                record.get("cache_memory_trace", {}).get(diagnostic)
                for record in condition_records
                if record.get("cache_memory_trace", {}).get(diagnostic) is not None
            ]
            row[diagnostic] = mean(values) if values else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values("token_f1", ascending=False)



SPLIT = "validation"
LIMIT = 50
RANDOM_SEED = 13
BASE_CONDITION = "dense_pool20_cross_top8_router"
DENSE_MODEL_NAME = "intfloat/e5-small-v2"
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"


ADAPTIVE_CONDITIONS = [
    "dense_pool20_cross_top8_router",
    "small80_pool20_cross_expand160_top8_router",
    "small80_pool20_cross_expand240_top8_router",
    "small80_pool30_cross_expand160_top8_router",
    "small80_cross_before_expand160_top8_router",
    "small80_expand160_cross_after_top8_router",
    "small80_cross_expand160_cross_top8_router",
    "sentence_pool30_cross_window_pm1_top8_router",
    "sentence_pool30_cross_window_pm2_top8_router",
    "sentence_pool50_cross_window_paragraph_top8_router",
    "paragraph_pool20_cross_top8_router",
    "paragraph_pool30_cross_top8_router",
    "small80_pool30_cross_expand160_budget600_router",
    "small80_pool30_cross_expand160_budget800_router",
    "sentence_pool50_cross_window_pm2_budget700_router",
]


def question_type_from_text(question: str) -> str:
    normalized = normalize_answer(question)
    tokens = normalized.split()
    if not tokens:
        return "free_form"
    first = tokens[0]
    if first in {"is", "are", "was", "were", "do", "does", "did", "can", "could", "will", "would", "has", "have", "had"}:
        return "yes_no"
    if first in {"what", "which", "who", "where", "when", "how"}:
        return "extractive"
    return "free_form"


def context_block(contexts: list[RetrievedContext], *, include_scores: bool = True) -> str:
    if not contexts:
        return "[No context provided]"
    lines = []
    for context in contexts:
        score_part = f" score={context.score:.4f}" if include_scores else ""
        lines.append(f"[{context.rank}] section={context.chunk.section}{score_part}\n{context.chunk.text}")
    return "\n\n".join(lines)


def build_question_type_router_prompt(question: str, contexts: list[RetrievedContext]) -> str:
    question_type = question_type_from_text(question)
    if question_type == "yes_no":
        instruction = "Return only Yes, No, or Unanswerable."
    elif question_type == "extractive":
        instruction = "Return only the shortest evidence-supported answer phrase."
    else:
        instruction = "Return one short evidence-supported answer sentence."
    return (
        "Answer using only the provided retrieved context. "
        f"Question type: {question_type}. {instruction}\n\n"
        f"Context:\n{context_block(contexts)}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def trim_first_line(text: str) -> str:
    text = str(text or "").strip()
    return text.splitlines()[0].strip() if text else ""


def source_items_for_paper(paper: Paper) -> list[dict[str, Any]]:
    items = []
    if paper.title:
        items.append({"label": "title", "text": paper.title, "kind": "title", "index": 0})
    if paper.abstract:
        items.append({"label": "abstract", "text": paper.abstract, "kind": "abstract", "index": 0})
    for section in paper.sections:
        for paragraph_index, paragraph in enumerate(section.paragraphs):
            label = f"{section.name} / paragraph {paragraph_index + 1}"
            items.append({"label": label, "text": paragraph, "kind": "paragraph", "index": paragraph_index})
    return items


def item_text_by_label(paper: Paper) -> dict[str, str]:
    return {item["label"]: item["text"] for item in source_items_for_paper(paper)}


def split_adaptive_sentences(text: str) -> list[str]:
    cleaned = re.sub(r"\s+", " ", str(text or "")).strip()
    if not cleaned:
        return []
    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", cleaned)
    sentences = []
    for piece in pieces:
        piece = piece.strip(" \t\r\n-")
        words = piece.split()
        if len(words) < 4:
            continue
        if len(words) > 90:
            for start in range(0, len(words), 70):
                span = " ".join(words[start:start + 90]).strip()
                if len(span.split()) >= 4:
                    sentences.append(span)
        else:
            sentences.append(piece)
    return sentences


def build_paragraph_chunks(paper: Paper) -> list[Chunk]:
    chunks = []
    for item in source_items_for_paper(paper):
        text = item["text"].strip()
        if not text:
            continue
        chunks.append(Chunk(
            paper_id=paper.paper_id,
            chunk_id=f"{paper.paper_id}:paragraph:{len(chunks)}",
            section=item["label"],
            text=text,
            title=paper.title,
            start_token=0,
            end_token=len(text.split()),
        ))
    return chunks


def build_sentence_chunks(paper: Paper) -> tuple[list[Chunk], dict[str, dict[str, Any]]]:
    chunks = []
    metadata = {}
    for item_index, item in enumerate(source_items_for_paper(paper)):
        token_cursor = 0
        for sentence_index, sentence in enumerate(split_adaptive_sentences(item["text"])):
            words = sentence.split()
            chunk_id = f"{paper.paper_id}:sentence:{item_index}:{sentence_index}"
            chunk = Chunk(
                paper_id=paper.paper_id,
                chunk_id=chunk_id,
                section=item["label"],
                text=sentence,
                title=paper.title,
                start_token=token_cursor,
                end_token=token_cursor + len(words),
            )
            chunks.append(chunk)
            metadata[chunk_id] = {
                "item_index": item_index,
                "sentence_index": sentence_index,
                "label": item["label"],
                "sentences": split_adaptive_sentences(item["text"]),
                "full_text": item["text"],
            }
            token_cursor += len(words)
    return chunks, metadata


def with_new_ranks(contexts: list[RetrievedContext], source: str | None = None) -> list[RetrievedContext]:
    return [
        RetrievedContext(context.chunk, float(context.score), rank, source or context.source)
        for rank, context in enumerate(contexts, start=1)
    ]


def dedupe_contexts_by_text(contexts: list[RetrievedContext]) -> list[RetrievedContext]:
    seen = set()
    deduped = []
    for context in contexts:
        key = normalize_answer(context.chunk.text)
        if not key:
            continue
        if key in seen:
            continue
        seen.add(key)
        deduped.append(context)
    return deduped


def expand_context_to_token_window(example: QAExample, context: RetrievedContext, target_tokens: int, source: str) -> RetrievedContext:
    text_by_label = item_text_by_label(example.paper)
    full_text = text_by_label.get(context.chunk.section, context.chunk.text)
    tokens = full_text.split()
    if not tokens:
        return context
    center = (int(context.chunk.start_token) + int(context.chunk.end_token)) // 2
    half = max(1, target_tokens // 2)
    start = max(0, center - half)
    end = min(len(tokens), start + target_tokens)
    start = max(0, end - target_tokens)
    expanded_text = " ".join(tokens[start:end]).strip()
    chunk = Chunk(
        paper_id=context.chunk.paper_id,
        chunk_id=f"{context.chunk.chunk_id}:{source}:{target_tokens}:{start}:{end}",
        section=context.chunk.section,
        text=expanded_text,
        title=context.chunk.title,
        start_token=start,
        end_token=end,
    )
    return RetrievedContext(chunk, context.score, context.rank, source)


def expand_sentence_window(example: QAExample, context: RetrievedContext, sentence_metadata: dict[str, dict[str, Any]], radius: int, source: str) -> RetrievedContext:
    info = sentence_metadata.get(context.chunk.chunk_id)
    if not info:
        return expand_context_to_token_window(example, context, 160, source)
    sentences = info["sentences"]
    sentence_index = info["sentence_index"]
    start = max(0, sentence_index - radius)
    end = min(len(sentences), sentence_index + radius + 1)
    text = " ".join(sentences[start:end]).strip()
    chunk = Chunk(
        paper_id=context.chunk.paper_id,
        chunk_id=f"{context.chunk.chunk_id}:{source}:pm{radius}",
        section=context.chunk.section,
        text=text,
        title=context.chunk.title,
        start_token=0,
        end_token=len(text.split()),
    )
    return RetrievedContext(chunk, context.score, context.rank, source)


def expand_sentence_to_paragraph(example: QAExample, context: RetrievedContext, source: str) -> RetrievedContext:
    text = item_text_by_label(example.paper).get(context.chunk.section, context.chunk.text)
    chunk = Chunk(
        paper_id=context.chunk.paper_id,
        chunk_id=f"{context.chunk.chunk_id}:{source}:paragraph",
        section=context.chunk.section,
        text=text,
        title=context.chunk.title,
        start_token=0,
        end_token=len(text.split()),
    )
    return RetrievedContext(chunk, context.score, context.rank, source)


def cross_rerank_top(question: str, contexts: list[RetrievedContext], final_top_k: int, source: str) -> list[RetrievedContext]:
    if not contexts:
        return []
    reranked = rerank_with_cross_encoder(question, contexts, CROSS_ENCODER_MODEL_NAME)
    return with_new_ranks(reranked[:final_top_k], source)


def pack_context_budget(contexts: list[RetrievedContext], budget_words: int, source: str) -> list[RetrievedContext]:
    packed = []
    used_words = 0
    for context in contexts:
        words = context.chunk.text.split()
        remaining = budget_words - used_words
        if remaining <= 0:
            break
        if len(words) > remaining:
            if not packed and remaining >= 60:
                text = " ".join(words[:remaining])
                chunk = Chunk(
                    paper_id=context.chunk.paper_id,
                    chunk_id=f"{context.chunk.chunk_id}:{source}:trimmed",
                    section=context.chunk.section,
                    text=text,
                    title=context.chunk.title,
                    start_token=context.chunk.start_token,
                    end_token=context.chunk.start_token + len(text.split()),
                )
                packed.append(RetrievedContext(chunk, context.score, context.rank, source))
                used_words += len(text.split())
            continue
        packed.append(RetrievedContext(context.chunk, context.score, context.rank, source))
        used_words += len(words)
    return with_new_ranks(packed, source)


def context_word_count(contexts: list[RetrievedContext]) -> int:
    return sum(len(context.chunk.text.split()) for context in contexts)


def adaptive_trace(
    *,
    condition: str,
    question: str,
    contexts: list[RetrievedContext],
    prompt: str,
    retrieval_unit: str,
    expansion_policy: str,
    rerank_stage: str,
    packing_budget_words: int,
    candidate_pool_k: int,
    final_top_k: int,
) -> dict[str, Any]:
    return {
        "strategy_family": "adaptive_context",
        "base_condition": BASE_CONDITION,
        "condition": condition,
        "retrieval_unit": retrieval_unit,
        "expansion_policy": expansion_policy,
        "rerank_stage": rerank_stage,
        "packing_budget_words": packing_budget_words,
        "candidate_pool_k": candidate_pool_k,
        "final_top_k": final_top_k,
        "reranker_model": CROSS_ENCODER_MODEL_NAME,
        "context_order": "cross_encoder_score",
        "prompt_variant": "question_type_router",
        "question_type": question_type_from_text(question),
        "retrieved_context_count": len(contexts),
        "context_word_count": context_word_count(contexts),
        "avg_context_score": mean(context.score for context in contexts) if contexts else 0.0,
        "max_context_score": max((context.score for context in contexts), default=0.0),
        "prompt_word_count": len(prompt.split()),
    }


def fixed_baseline_contexts(question: str, chunks160: list[Chunk]) -> tuple[list[RetrievedContext], dict[str, Any]]:
    pool = retrieve_dense(question, chunks160, 20, model_name=DENSE_MODEL_NAME)
    contexts = cross_rerank_top(question, pool, 8, "dense_pool20_cross")
    return contexts, {
        "retrieval_unit": "fixed_chunk",
        "expansion_policy": "none",
        "rerank_stage": "after_retrieval",
        "packing_budget_words": 0,
        "candidate_pool_k": 20,
        "final_top_k": 8,
    }


def small_to_large_contexts(example: QAExample, condition: str) -> tuple[list[RetrievedContext], dict[str, Any]]:
    pool_k = 30 if "pool30" in condition else 20
    expand_tokens = 240 if "expand240" in condition else 160
    budget = 0
    if "budget600" in condition:
        budget = 600
    if "budget800" in condition:
        budget = 800

    small_chunks = chunk_paper(example.paper, chunk_size_tokens=80, chunk_overlap_tokens=20)
    pool = retrieve_dense(example.question, small_chunks, pool_k, model_name=DENSE_MODEL_NAME)

    if condition == "small80_expand160_cross_after_top8_router":
        expanded_pool = dedupe_contexts_by_text([
            expand_context_to_token_window(example, context, expand_tokens, "small80_expand160")
            for context in pool
        ])
        contexts = cross_rerank_top(example.question, expanded_pool, 8, "small80_expand160_cross_after")
        rerank_stage = "after_expansion"
    elif condition == "small80_cross_expand160_cross_top8_router":
        first_pass = cross_rerank_top(example.question, pool, min(16, len(pool)), "small80_cross_before")
        expanded_pool = dedupe_contexts_by_text([
            expand_context_to_token_window(example, context, expand_tokens, "small80_cross_expand160")
            for context in first_pass
        ])
        contexts = cross_rerank_top(example.question, expanded_pool, 8, "small80_cross_expand160_cross")
        rerank_stage = "before_and_after"
    else:
        first_pass = cross_rerank_top(example.question, pool, 8 if budget == 0 else min(14, len(pool)), "small80_cross_before")
        expanded = dedupe_contexts_by_text([
            expand_context_to_token_window(example, context, expand_tokens, f"small80_expand{expand_tokens}")
            for context in first_pass
        ])
        contexts = with_new_ranks(expanded[:8 if budget == 0 else len(expanded)], f"small80_cross_expand{expand_tokens}")
        rerank_stage = "before_expansion"

    if budget:
        contexts = pack_context_budget(contexts, budget, f"budget{budget}")

    return contexts, {
        "retrieval_unit": "small_chunk",
        "expansion_policy": "budget_pack" if budget else "parent_chunk",
        "rerank_stage": rerank_stage,
        "packing_budget_words": budget,
        "candidate_pool_k": pool_k,
        "final_top_k": len(contexts),
    }


def sentence_contexts(example: QAExample, condition: str) -> tuple[list[RetrievedContext], dict[str, Any]]:
    pool_k = 50 if "pool50" in condition else 30
    budget = 700 if "budget700" in condition else 0
    sentence_chunks, sentence_metadata = build_sentence_chunks(example.paper)
    pool = retrieve_dense(example.question, sentence_chunks, pool_k, model_name=DENSE_MODEL_NAME)
    first_pass = cross_rerank_top(example.question, pool, 8 if budget == 0 else min(16, len(pool)), "sentence_cross")

    if "paragraph" in condition:
        expanded = [
            expand_sentence_to_paragraph(example, context, "sentence_to_paragraph")
            for context in first_pass
        ]
        expansion_policy = "paragraph_window"
    else:
        radius = 2 if "pm2" in condition else 1
        expanded = [
            expand_sentence_window(example, context, sentence_metadata, radius, f"sentence_window_pm{radius}")
            for context in first_pass
        ]
        expansion_policy = "budget_pack" if budget else "sentence_window"

    contexts = with_new_ranks(dedupe_contexts_by_text(expanded)[:8 if budget == 0 else len(expanded)], "sentence_expanded")
    if budget:
        contexts = pack_context_budget(contexts, budget, f"sentence_budget{budget}")

    return contexts, {
        "retrieval_unit": "sentence",
        "expansion_policy": expansion_policy,
        "rerank_stage": "before_expansion",
        "packing_budget_words": budget,
        "candidate_pool_k": pool_k,
        "final_top_k": len(contexts),
    }


def paragraph_contexts(example: QAExample, condition: str) -> tuple[list[RetrievedContext], dict[str, Any]]:
    pool_k = 30 if "pool30" in condition else 20
    paragraph_chunks = build_paragraph_chunks(example.paper)
    pool = retrieve_dense(example.question, paragraph_chunks, pool_k, model_name=DENSE_MODEL_NAME)
    contexts = cross_rerank_top(example.question, pool, 8, "paragraph_cross")
    return contexts, {
        "retrieval_unit": "paragraph",
        "expansion_policy": "none",
        "rerank_stage": "after_retrieval",
        "packing_budget_words": 0,
        "candidate_pool_k": pool_k,
        "final_top_k": 8,
    }


def adaptive_contexts_for_condition(example: QAExample, condition: str) -> tuple[list[RetrievedContext], dict[str, Any]]:
    chunks160 = chunk_paper(example.paper, chunk_size_tokens=160, chunk_overlap_tokens=30)
    if condition == "dense_pool20_cross_top8_router":
        return fixed_baseline_contexts(example.question, chunks160)
    if condition.startswith("small80_"):
        return small_to_large_contexts(example, condition)
    if condition.startswith("sentence_"):
        return sentence_contexts(example, condition)
    if condition.startswith("paragraph_"):
        return paragraph_contexts(example, condition)
    raise ValueError(condition)


def run_adaptive_condition(example: QAExample, condition: str) -> dict[str, Any]:
    contexts, config = adaptive_contexts_for_condition(example, condition)
    prompt = build_question_type_router_prompt(example.question, contexts)
    raw_prediction = generate_text(prompt)
    prediction = trim_first_line(raw_prediction)
    context_texts = [context.chunk.text for context in contexts]
    trace = adaptive_trace(
        condition=condition,
        question=example.question,
        contexts=contexts,
        prompt=prompt,
        retrieval_unit=config["retrieval_unit"],
        expansion_policy=config["expansion_policy"],
        rerank_stage=config["rerank_stage"],
        packing_budget_words=config["packing_budget_words"],
        candidate_pool_k=config["candidate_pool_k"],
        final_top_k=config["final_top_k"],
    )
    return {
        "example_id": example.example_id,
        "condition": condition,
        "question": example.question,
        "gold_answers": example.gold_texts(),
        "prediction": prediction,
        "raw_prediction": raw_prediction,
        "metrics": score_prediction(prediction, example, context_texts),
        "retrieved_contexts": [context.as_dict() for context in contexts],
        "ablation_trace": trace,
    }


def summarize_adaptive(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for condition in sorted({record["condition"] for record in records}):
        condition_records = [record for record in records if record["condition"] == condition]
        row = {"condition": condition, "count": float(len(condition_records))}
        for metric in ["token_f1", "exact_match", "context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
            row[metric] = mean(record["metrics"][metric] for record in condition_records)
        for diagnostic in [
            "candidate_pool_k",
            "final_top_k",
            "packing_budget_words",
            "retrieved_context_count",
            "context_word_count",
            "avg_context_score",
            "max_context_score",
            "prompt_word_count",
        ]:
            row[diagnostic] = mean(float(record["ablation_trace"][diagnostic]) for record in condition_records)
        first_trace = condition_records[0]["ablation_trace"]
        for label in [
            "strategy_family",
            "base_condition",
            "retrieval_unit",
            "expansion_policy",
            "rerank_stage",
            "reranker_model",
            "context_order",
            "prompt_variant",
        ]:
            row[label] = first_trace[label]
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["token_f1", "context_recall"], ascending=False)


def trace_frame_from_records(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        trace = record["ablation_trace"]
        rows.append({
            "example_id": record["example_id"],
            "condition": record["condition"],
            "question_type": trace["question_type"],
            "retrieval_unit": trace["retrieval_unit"],
            "expansion_policy": trace["expansion_policy"],
            "rerank_stage": trace["rerank_stage"],
            "candidate_pool_k": trace["candidate_pool_k"],
            "final_top_k": trace["final_top_k"],
            "packing_budget_words": trace["packing_budget_words"],
            "context_word_count": trace["context_word_count"],
            "prompt_word_count": trace["prompt_word_count"],
            "token_f1": record["metrics"]["token_f1"],
            "exact_match": record["metrics"]["exact_match"],
            "context_precision": record["metrics"]["context_precision"],
            "context_recall": record["metrics"]["context_recall"],
            "prediction": record["prediction"],
            "gold_answers": record["gold_answers"],
        })
    return pd.DataFrame(rows)


def display_adaptive_outputs(records: list[dict[str, Any]]) -> pd.DataFrame:
    summary = summarize_adaptive(records)
    print("ADAPTIVE_CHUNK_FLEXIBLE_CONTEXT_FINAL_METRICS")
    display(summary)

    print("BEST_BY_STRATEGY")
    display(
        summary.sort_values(["retrieval_unit", "expansion_policy", "token_f1", "context_recall"], ascending=[True, True, False, False])
        .groupby(["retrieval_unit", "expansion_policy"], as_index=False)
        .head(2)
        .sort_values(["token_f1", "context_recall"], ascending=False)
    )

    trace_frame = trace_frame_from_records(records)
    print("QUESTION_TYPE_SUMMARY")
    display(
        trace_frame.groupby(["condition", "question_type"], as_index=False)
        .agg(
            count=("example_id", "count"),
            token_f1=("token_f1", "mean"),
            exact_match=("exact_match", "mean"),
            context_precision=("context_precision", "mean"),
            context_recall=("context_recall", "mean"),
        )
        .sort_values(["condition", "question_type"])
    )

    print("CONTEXT_BUDGET_WORD_COUNT_SUMMARY")
    display(
        trace_frame.groupby(["condition", "retrieval_unit", "expansion_policy", "packing_budget_words"], as_index=False)
        .agg(
            context_word_count=("context_word_count", "mean"),
            prompt_word_count=("prompt_word_count", "mean"),
            final_top_k=("final_top_k", "mean"),
            token_f1=("token_f1", "mean"),
            context_recall=("context_recall", "mean"),
        )
        .sort_values(["token_f1", "context_recall"], ascending=False)
    )

    print("CSV_READY_ROWS")
    display(summary[[
        "condition",
        "count",
        "token_f1",
        "exact_match",
        "context_precision",
        "context_recall",
        "faithfulness",
        "answer_relevancy",
        "candidate_pool_k",
        "final_top_k",
        "context_word_count",
        "retrieval_unit",
        "expansion_policy",
        "rerank_stage",
        "packing_budget_words",
        "reranker_model",
        "prompt_variant",
    ]])

    print("SAMPLE_RECORD_PREVIEW")
    display(trace_frame.head(80))
    return summary



BASE_CONDITION = "small80_pool30_cross_expand160_budget800_router"

COMPOSER_CONDITIONS = [
    "best_adaptive_direct_router",
    "best_adaptive_slm_extract_then_slm_answer",
    "best_adaptive_slm_extract_then_rule_compose",
    "best_adaptive_ir_extract_then_rule_compose",
    "best_adaptive_hybrid_extract_then_rule_compose",
    "dense_pool20_cross_slm_extract_then_rule_compose",
    "oracle_evidence_slm_extract_then_rule_compose",
]

NEGATION_RE = re.compile(r"\b(no|not|never|none|neither|cannot|can't|does not|do not|did not|without|fail|fails|failed|unanswerable|unknown)\b", re.I)
AFFIRMATION_RE = re.compile(r"\b(yes|supports|show|shows|demonstrate|demonstrates|use|uses|used|include|includes|included|is|are|was|were|can|could|does|do)\b", re.I)
QUESTION_STOPWORDS = {
    "what", "which", "who", "where", "when", "how", "why", "does", "do", "did", "is", "are", "was", "were",
    "can", "could", "will", "would", "the", "a", "an", "of", "to", "in", "for", "on", "and", "or", "with",
    "this", "that", "these", "those", "paper", "study", "authors", "author"
}


def answer_type_from_example(example: QAExample) -> str:
    question_type = question_type_from_text(example.question)
    if any(answer.unanswerable for answer in example.answers):
        return "unanswerable"
    if question_type == "yes_no":
        return "yes_no"
    if any(answer.extractive_spans for answer in example.answers):
        return "extractive"
    return "free_form"


def content_token_set(text: str) -> set[str]:
    return {token for token in tokenize(text) if token not in QUESTION_STOPWORDS and len(token) > 2}


def evidence_overlap_score(question: str, sentence: str, source_score: float = 0.0, rank: int = 1) -> float:
    q_tokens = content_token_set(question)
    s_tokens = content_token_set(sentence)
    lexical = len(q_tokens & s_tokens) / max(1, len(q_tokens))
    q_entities = set(re.findall(r"\b[A-Z][A-Za-z0-9-]{2,}\b|\b\d+(?:\.\d+)?%?\b", question))
    s_entities = set(re.findall(r"\b[A-Z][A-Za-z0-9-]{2,}\b|\b\d+(?:\.\d+)?%?\b", sentence))
    entity = len(q_entities & s_entities) / max(1, len(q_entities)) if q_entities else 0.0
    number_bonus = 0.08 if re.search(r"\d", question) and re.search(r"\d", sentence) else 0.0
    rank_bonus = 1.0 / max(2.0, rank + 1.0)
    return 0.52 * lexical + 0.18 * entity + 0.18 * float(source_score) + 0.07 * rank_bonus + number_bonus


def evidence_precision_recall(selected_texts: list[str], gold_texts: list[str]) -> tuple[float, float]:
    selected = [normalize_answer(text) for text in selected_texts if normalize_answer(text)]
    gold = [normalize_answer(text) for text in gold_texts if normalize_answer(text)]
    if not selected:
        precision = 1.0 if not gold else 0.0
    else:
        selected_hits = 0
        for selected_text in selected:
            if any(selected_text in gold_text or gold_text in selected_text for gold_text in gold):
                selected_hits += 1
        precision = selected_hits / max(1, len(selected))
    if not gold:
        recall = 1.0
    else:
        gold_hits = 0
        for gold_text in gold:
            if any(selected_text in gold_text or gold_text in selected_text for selected_text in selected):
                gold_hits += 1
        recall = gold_hits / max(1, len(gold))
    return float(precision), float(recall)


def evidence_dict(text: str, score: float, source: str, context_rank: int = 0, sentence_index: int = 0) -> dict[str, Any]:
    return {
        "text": " ".join(str(text or "").split()).strip(),
        "score": float(score),
        "source": source,
        "context_rank": int(context_rank),
        "sentence_index": int(sentence_index),
    }


def split_composer_sentences(text: str) -> list[str]:
    cleaned = re.sub(r"\s+", " ", str(text or "")).strip()
    if not cleaned:
        return []
    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", cleaned)
    sentences = []
    for piece in pieces:
        piece = piece.strip(" \t\r\n-")
        words = piece.split()
        if len(words) < 4:
            continue
        if len(words) > 90:
            for start in range(0, len(words), 70):
                span = " ".join(words[start:start + 90]).strip()
                if len(span.split()) >= 4:
                    sentences.append(span)
        else:
            sentences.append(piece)
    return sentences


def ir_extract_evidence(question: str, contexts: list[RetrievedContext], max_evidence: int = 5) -> list[dict[str, Any]]:
    rows = []
    for context in contexts:
        for sentence_index, sentence in enumerate(split_composer_sentences(context.chunk.text), start=1):
            score = evidence_overlap_score(question, sentence, context.score, context.rank)
            rows.append(evidence_dict(sentence, score, "ir_rule", context.rank, sentence_index))
    rows.sort(key=lambda row: row["score"], reverse=True)
    selected = []
    seen = set()
    per_context_counts: dict[int, int] = {}
    for row in rows:
        key = normalize_answer(row["text"])
        if not key or key in seen:
            continue
        if row["score"] < 0.08 and selected:
            continue
        context_rank = row["context_rank"]
        if per_context_counts.get(context_rank, 0) >= 2:
            continue
        selected.append(row)
        seen.add(key)
        per_context_counts[context_rank] = per_context_counts.get(context_rank, 0) + 1
        if len(selected) >= max_evidence:
            break
    return selected


def build_slm_extraction_prompt(example: QAExample, contexts: list[RetrievedContext]) -> str:
    return (
        "Select the minimal evidence needed to answer the question. "
        "Return at most 5 short evidence lines copied or closely paraphrased from the context. "
        "Do not answer the question yet. If no evidence supports an answer, return Unanswerable.\n\n"
        f"Question type: {answer_type_from_example(example)}\n\n"
        f"Context:\n{context_block(contexts)}\n\n"
        f"Question: {example.question}\n"
        "Evidence:\n"
        "1."
    )


def parse_slm_evidence(raw_text: str, question: str, contexts: list[RetrievedContext], max_evidence: int = 5) -> list[dict[str, Any]]:
    text = str(raw_text or "").strip()
    if not text:
        return []
    if re.search(r"\bunanswerable\b", text, flags=re.I):
        return [evidence_dict("Unanswerable", 0.0, "slm", 0, 0)]
    lines = []
    for line in text.splitlines():
        cleaned = re.sub(r"^\s*(?:[-*]|\d+[.)]|evidence\s*\d*\s*:)\s*", "", line, flags=re.I).strip()
        cleaned = cleaned.strip(" \"'")
        if not cleaned or cleaned.lower().startswith(("question:", "answer:", "final answer")):
            continue
        lines.append(cleaned)
    if not lines:
        lines = split_composer_sentences(text)
    rows = []
    for index, line in enumerate(lines[:max_evidence], start=1):
        score = evidence_overlap_score(question, line, 0.0, index)
        rows.append(evidence_dict(line, score, "slm", 0, index))
    if rows:
        return rows
    return ir_extract_evidence(question, contexts, max_evidence=max_evidence)


def slm_extract_evidence(example: QAExample, contexts: list[RetrievedContext], max_evidence: int = 5) -> tuple[list[dict[str, Any]], str]:
    prompt = build_slm_extraction_prompt(example, contexts)
    raw = generate_text(prompt)
    return parse_slm_evidence(raw, example.question, contexts, max_evidence=max_evidence), raw


def merge_evidence_lists(first: list[dict[str, Any]], second: list[dict[str, Any]], max_evidence: int = 5) -> list[dict[str, Any]]:
    merged = []
    seen = set()
    for row in first + second:
        key = normalize_answer(row["text"])
        if not key or key in seen:
            continue
        merged.append(row)
        seen.add(key)
        if len(merged) >= max_evidence:
            break
    return merged


def evidence_contexts_from_rows(example: QAExample, evidence_rows: list[dict[str, Any]], source: str) -> list[RetrievedContext]:
    contexts = []
    for index, row in enumerate(evidence_rows, start=1):
        text = row["text"]
        chunk = Chunk(
            paper_id=example.paper_id,
            chunk_id=f"{example.paper_id}:{example.question_id}:{source}:{index}",
            section=f"{source}_evidence",
            text=text,
            title=example.paper.title,
            start_token=0,
            end_token=len(text.split()),
        )
        contexts.append(RetrievedContext(chunk, float(row.get("score", 0.0)), index, source))
    return contexts


def build_slm_answer_from_evidence_prompt(example: QAExample, evidence_rows: list[dict[str, Any]]) -> str:
    evidence_text = "\n".join(f"[E{index}] {row['text']}" for index, row in enumerate(evidence_rows, start=1)) or "[No evidence]"
    answer_type = answer_type_from_example(example)
    if answer_type == "yes_no":
        instruction = "Return only Yes, No, or Unanswerable."
    elif answer_type == "extractive":
        instruction = "Return only the shortest answer phrase supported by the evidence."
    else:
        instruction = "Return one short answer sentence supported by the evidence."
    return (
        "Answer using only the extracted evidence. "
        f"{instruction}\n\n"
        f"Evidence:\n{evidence_text}\n\n"
        f"Question: {example.question}\n"
        "Answer:"
    )


def yes_no_rule(question: str, evidence_rows: list[dict[str, Any]]) -> str:
    joined = " ".join(row["text"] for row in evidence_rows)
    if not joined.strip() or normalize_answer(joined) == "unanswerable":
        return "Unanswerable"
    if NEGATION_RE.search(joined):
        return "No"
    if AFFIRMATION_RE.search(joined) or content_token_set(question) & content_token_set(joined):
        return "Yes"
    return "Unanswerable"


def shortest_informative_span(question: str, evidence_rows: list[dict[str, Any]], max_words: int = 18) -> str:
    q_tokens = content_token_set(question)
    candidates = []
    for row in evidence_rows:
        text = row["text"]
        for sentence in split_composer_sentences(text) or [text]:
            words = sentence.split()
            if not words:
                continue
            score = evidence_overlap_score(question, sentence, row.get("score", 0.0), row.get("context_rank", 1))
            entity_or_number = re.findall(r"\b[A-Z][A-Za-z0-9-]{2,}(?:\s+[A-Z][A-Za-z0-9-]{2,}){0,4}\b|\b\d+(?:\.\d+)?%?\b", sentence)
            if entity_or_number:
                span = " ".join(entity_or_number[:4])
            else:
                filtered = [word.strip(" ,.;:()[]") for word in words if normalize_answer(word) not in q_tokens]
                span = " ".join(filtered[:max_words]) or " ".join(words[:max_words])
            candidates.append((score, len(span.split()), span.strip(" ,.;:")))
    candidates = [(score, length, span) for score, length, span in candidates if span]
    if not candidates:
        return "Unanswerable"
    candidates.sort(key=lambda item: (-item[0], item[1]))
    return candidates[0][2]


def rule_compose_answer(example: QAExample, evidence_rows: list[dict[str, Any]]) -> str:
    answer_type = answer_type_from_example(example)
    filtered = [row for row in evidence_rows if normalize_answer(row["text"]) != "unanswerable"]
    if not filtered:
        return "Unanswerable"
    max_score = max((row.get("score", 0.0) for row in filtered), default=0.0)
    if max_score < 0.07 and answer_type != "free_form":
        return "Unanswerable"
    if answer_type == "yes_no":
        return yes_no_rule(example.question, filtered)
    if answer_type == "extractive":
        return shortest_informative_span(example.question, filtered)
    facts = []
    seen = set()
    for row in sorted(filtered, key=lambda item: item.get("score", 0.0), reverse=True):
        sentence = (split_composer_sentences(row["text"]) or [row["text"]])[0]
        sentence = sentence.strip()
        key = normalize_answer(sentence)
        if not key or key in seen:
            continue
        facts.append(sentence)
        seen.add(key)
        if len(facts) >= 2:
            break
    return " ".join(facts).strip() or shortest_informative_span(example.question, filtered, max_words=24)


def oracle_evidence_contexts(example: QAExample) -> list[RetrievedContext]:
    contexts = []
    for index, text in enumerate(example.evidence_texts(), start=1):
        chunk = Chunk(
            paper_id=example.paper_id,
            chunk_id=f"{example.paper_id}:oracle_evidence:{example.question_id}:{index}",
            section="gold_evidence",
            text=text,
            title=example.paper.title,
            start_token=0,
            end_token=len(text.split()),
        )
        contexts.append(RetrievedContext(chunk, 1.0, index, "oracle_evidence"))
    return contexts or []


def contexts_for_composer_condition(example: QAExample, condition: str) -> tuple[list[RetrievedContext], str]:
    if condition.startswith("dense_pool20_cross"):
        contexts, _ = adaptive_contexts_for_condition(example, "dense_pool20_cross_top8_router")
        return contexts, "dense_pool20_cross_top8_router"
    if condition.startswith("oracle_evidence"):
        return oracle_evidence_contexts(example), "oracle_evidence"
    contexts, _ = adaptive_contexts_for_condition(example, "small80_pool30_cross_expand160_budget800_router")
    return contexts, "small80_pool30_cross_expand160_budget800_router"


def extraction_for_condition(example: QAExample, condition: str, contexts: list[RetrievedContext]) -> tuple[list[dict[str, Any]], str, str]:
    if condition == "best_adaptive_ir_extract_then_rule_compose":
        return ir_extract_evidence(example.question, contexts), "", "ir_rule"
    if condition == "best_adaptive_hybrid_extract_then_rule_compose":
        slm_rows, raw = slm_extract_evidence(example, contexts)
        ir_rows = ir_extract_evidence(example.question, contexts)
        return merge_evidence_lists(slm_rows, ir_rows), raw, "hybrid"
    rows, raw = slm_extract_evidence(example, contexts)
    return rows, raw, "slm"


def error_tag_for_record(record: dict[str, Any]) -> str:
    trace = record["ablation_trace"]
    token_f1 = record["metrics"]["token_f1"]
    if trace["extraction_recall"] < 0.25:
        return "low_extraction_recall"
    if trace["extraction_precision"] < 0.25:
        return "noisy_extraction"
    if token_f1 < 0.25:
        return "composition_or_wording_error"
    return "ok_or_partial"


def run_composer_condition(example: QAExample, condition: str) -> dict[str, Any]:
    contexts, context_source = contexts_for_composer_condition(example, condition)
    raw_extraction = ""
    evidence_rows: list[dict[str, Any]] = []
    composer_source = "direct_slm"

    if condition == "best_adaptive_direct_router":
        prompt = build_question_type_router_prompt(example.question, contexts)
        raw_prediction = generate_text(prompt)
        prediction = trim_first_line(raw_prediction)
        evidence_rows = ir_extract_evidence(example.question, contexts)
        selected_contexts = contexts
        composer_source = "direct_slm"
    else:
        evidence_rows, raw_extraction, extractor_source = extraction_for_condition(example, condition, contexts)
        selected_contexts = evidence_contexts_from_rows(example, evidence_rows, extractor_source)
        if condition == "best_adaptive_slm_extract_then_slm_answer":
            prompt = build_slm_answer_from_evidence_prompt(example, evidence_rows)
            raw_prediction = generate_text(prompt)
            prediction = trim_first_line(raw_prediction)
            composer_source = "slm"
        else:
            prompt = ""
            raw_prediction = rule_compose_answer(example, evidence_rows)
            prediction = raw_prediction
            composer_source = extractor_source if extractor_source != "slm" else "slm_rule"

    context_texts = [context.chunk.text for context in selected_contexts]
    evidence_texts = [row["text"] for row in evidence_rows]
    extraction_precision, extraction_recall = evidence_precision_recall(evidence_texts, example.evidence_texts())
    metrics = score_prediction(prediction, example, context_texts)
    trace = {
        "strategy_family": "intermediate_extraction_composer",
        "base_condition": BASE_CONDITION,
        "context_source": context_source,
        "answer_type": answer_type_from_example(example),
        "composer_source": composer_source,
        "selected_evidence_count": len(evidence_rows),
        "evidence_score": mean(row.get("score", 0.0) for row in evidence_rows) if evidence_rows else 0.0,
        "extraction_precision": extraction_precision,
        "extraction_recall": extraction_recall,
        "answer_token_f1": metrics["token_f1"],
        "exact_match": metrics["exact_match"],
        "context_word_count": sum(len(text.split()) for text in context_texts),
        "prompt_word_count": len(prompt.split()) if prompt else 0,
        "question_type": question_type_from_text(example.question),
    }
    record = {
        "example_id": example.example_id,
        "condition": condition,
        "question": example.question,
        "gold_answers": example.gold_texts(),
        "prediction": prediction,
        "raw_prediction": raw_prediction,
        "raw_extraction": raw_extraction,
        "selected_evidence": evidence_rows,
        "metrics": metrics,
        "retrieved_contexts": [context.as_dict() for context in selected_contexts],
        "ablation_trace": trace,
    }
    record["error_tag"] = error_tag_for_record(record)
    return record


def summarize_composer(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for condition in sorted({record["condition"] for record in records}):
        condition_records = [record for record in records if record["condition"] == condition]
        row = {"condition": condition, "count": float(len(condition_records))}
        for metric in ["token_f1", "exact_match", "context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
            row[metric] = mean(record["metrics"][metric] for record in condition_records)
        for diagnostic in [
            "extraction_precision",
            "extraction_recall",
            "selected_evidence_count",
            "evidence_score",
            "context_word_count",
            "prompt_word_count",
        ]:
            row[diagnostic] = mean(float(record["ablation_trace"][diagnostic]) for record in condition_records)
        first_trace = condition_records[0]["ablation_trace"]
        for label in ["strategy_family", "base_condition", "context_source", "composer_source"]:
            row[label] = first_trace[label]
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["token_f1", "extraction_recall"], ascending=False)


def composer_trace_frame(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        trace = record["ablation_trace"]
        rows.append({
            "example_id": record["example_id"],
            "condition": record["condition"],
            "answer_type": trace["answer_type"],
            "question_type": trace["question_type"],
            "composer_source": trace["composer_source"],
            "selected_evidence_count": trace["selected_evidence_count"],
            "extraction_precision": trace["extraction_precision"],
            "extraction_recall": trace["extraction_recall"],
            "token_f1": record["metrics"]["token_f1"],
            "exact_match": record["metrics"]["exact_match"],
            "context_precision": record["metrics"]["context_precision"],
            "context_recall": record["metrics"]["context_recall"],
            "error_tag": record["error_tag"],
            "prediction": record["prediction"],
            "gold_answers": record["gold_answers"],
            "selected_evidence": [row["text"] for row in record["selected_evidence"]],
        })
    return pd.DataFrame(rows)


def display_composer_outputs(records: list[dict[str, Any]]) -> pd.DataFrame:
    summary = summarize_composer(records)
    print("INTERMEDIATE_EXTRACTION_COMPOSER_FINAL_METRICS")
    display(summary)

    print("EXTRACTION_PRECISION_RECALL_TABLE")
    display(summary[[
        "condition",
        "context_source",
        "composer_source",
        "extraction_precision",
        "extraction_recall",
        "selected_evidence_count",
        "token_f1",
        "exact_match",
    ]].sort_values(["extraction_recall", "token_f1"], ascending=False))

    trace_frame = composer_trace_frame(records)
    print("ANSWER_TYPE_SUMMARY")
    display(
        trace_frame.groupby(["condition", "answer_type"], as_index=False)
        .agg(
            count=("example_id", "count"),
            token_f1=("token_f1", "mean"),
            exact_match=("exact_match", "mean"),
            extraction_precision=("extraction_precision", "mean"),
            extraction_recall=("extraction_recall", "mean"),
        )
        .sort_values(["condition", "answer_type"])
    )

    print("DIRECT_VS_EXTRACT_VS_COMPOSE")
    display(
        summary.assign(
            route=summary["condition"].map(lambda value: "direct" if "direct" in value else "extract_then_slm" if "slm_answer" in value else "extract_then_rule")
        )[["route", "condition", "token_f1", "exact_match", "extraction_precision", "extraction_recall", "context_recall"]]
        .sort_values(["route", "token_f1"], ascending=[True, False])
    )

    print("PER_EXAMPLE_ERROR_TAGS")
    display(
        trace_frame.groupby(["condition", "error_tag"], as_index=False)
        .agg(count=("example_id", "count"), token_f1=("token_f1", "mean"))
        .sort_values(["condition", "count"], ascending=[True, False])
    )

    print("CSV_READY_ROWS")
    display(summary[[
        "condition",
        "count",
        "token_f1",
        "exact_match",
        "context_precision",
        "context_recall",
        "faithfulness",
        "answer_relevancy",
        "extraction_precision",
        "extraction_recall",
        "selected_evidence_count",
        "context_source",
        "composer_source",
    ]])

    print("SAMPLE_RECORD_PREVIEW")
    display(trace_frame.head(80))
    return summary


## Run Experiment

Runs direct adaptive baseline, SLM extraction to SLM answer, SLM/IR/hybrid extraction to rule composer, dense rerank extraction, and oracle-evidence extraction diagnostics.

In [3]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

examples = load_qasper_examples(SPLIT, limit=LIMIT)
print(f"Loaded {len(examples)} Qasper QA examples")
print(f"Intermediate extraction/composer conditions: {len(COMPOSER_CONDITIONS)}")
print(f"Expected records: {len(examples) * len(COMPOSER_CONDITIONS)}")
print("Best realistic reference: small80_pool30_cross_expand160_budget800_router token_f1=0.319529.")
print("Stable rerank reference: dense_pool20_cross_top8_router token_f1=0.313624.")
print("Oracle evidence-only router reference: token_f1=0.312332.")
print("Oracle extract-then-answer upper diagnostic: token_f1=0.510129.")

records = []
for index, example in enumerate(examples, start=1):
    print(f"[{index}/{len(examples)}] {example.example_id}")
    for condition in COMPOSER_CONDITIONS:
        records.append(run_composer_condition(example, condition))

print(f"Final record count: {len(records)}")
summary = display_composer_outputs(records)


Loaded 50 Qasper QA examples
Intermediate extraction/composer conditions: 7
Expected records: 350
Best realistic reference: small80_pool30_cross_expand160_budget800_router token_f1=0.319529.
Stable rerank reference: dense_pool20_cross_top8_router token_f1=0.313624.
Oracle evidence-only router reference: token_f1=0.312332.
Oracle extract-then-answer upper diagnostic: token_f1=0.510129.
[1/50] 1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-06-02 15:14:28.796536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780413268.997973      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780413269.056368      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780413269.544136      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

,condition,error_tag,count,token_f1
0,best_adaptive_direct_router,noisy_extraction,50,0.319529
1,best_adaptive_hybrid_extract_then_rule_compose,noisy_extraction,50,0.193818
2,best_adaptive_ir_extract_then_rule_compose,noisy_extraction,50,0.172104
3,best_adaptive_slm_extract_then_rule_compose,noisy_extraction,50,0.131586
4,best_adaptive_slm_extract_then_slm_answer,noisy_extraction,50,0.120964
5,dense_pool20_cross_slm_extract_then_rule_compose,noisy_extraction,50,0.129270
6,oracle_evidence_slm_extract_then_rule_compose,noisy_extraction,50,0.100000


CSV_READY_ROWS


,condition,count,token_f1,exact_match,context_precision,context_recall,faithfulness,answer_relevancy,extraction_precision,extraction_recall,selected_evidence_count,context_source,composer_source
0,best_adaptive_direct_router,50.0,0.319529,0.14,0.220276,0.557063,0.766788,0.590067,0.0,1.0,3.10,small80_pool30_cross_expand160_budget800_router,direct_slm
1,best_adaptive_hybrid_extract_then_rule_compose,50.0,0.193818,0.12,0.144333,0.262222,0.766000,0.268906,0.0,1.0,3.74,small80_pool30_cross_expand160_budget800_router,hybrid
2,best_adaptive_ir_extract_then_rule_compose,50.0,0.172104,0.12,0.164667,0.220000,0.546000,0.235763,0.0,1.0,3.10,small80_pool30_cross_expand160_budget800_router,ir_rule
3,best_adaptive_slm_extract_then_rule_compose,50.0,0.131586,0.08,0.100000,0.072222,0.811667,0.194939,0.0,1.0,1.20,small80_pool30_cross_expand160_budget800_router,slm_rule
5,dense_pool20_cross_slm_extract_then_rule_compose,50.0,0.129270,0.08,0.155000,0.105714,0.810000,0.207338,0.0,1.0,1.26,dense_pool20_cross_top8_router,slm_rule
4,best_adaptive_slm_extract_then_slm_answer,50.0,0.120964,0.02,0.100000,0.072222,0.531783,0.338855,0.0,1.0,1.20,small80_pool30_cross_expand160_budget800_router,slm
6,oracle_evidence_slm_extract_then_rule_compose,50.0,0.100000,0.10,0.120000,0.073333,1.000000,0.100000,0.0,1.0,1.00,oracle_evidence,slm_rule


SAMPLE_RECORD_PREVIEW


,example_id,condition,answer_type,question_type,composer_source,selected_evidence_count,extraction_precision,extraction_recall,token_f1,exact_match,context_precision,context_recall,error_tag,prediction,gold_answers,selected_evidence
0,1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfd...,best_adaptive_direct_router,extractive,extractive,direct_slm,5,0.0,1.0,0.666667,0.0,0.333333,1.0,noisy_extraction,multilingual NMT,"[BIBREF19, BIBREF20, multilingual NMT (MNMT) B...",[We compare our approaches with related approa...
1,1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfd...,best_adaptive_slm_extract_then_slm_answer,extractive,extractive,slm,1,0.0,1.0,0.666667,0.0,0.000000,0.0,noisy_extraction,Multilingual NMT (MNMT) and BIBREF23,"[BIBREF19, BIBREF20, multilingual NMT (MNMT) B...",[The multilingual approaches compared with are...
2,1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfd...,best_adaptive_slm_extract_then_rule_compose,extractive,extractive,slm_rule,1,0.0,1.0,0.750000,0.0,0.000000,0.0,noisy_extraction,The Multilingual NMT MNMT BIBREF23,"[BIBREF19, BIBREF20, multilingual NMT (MNMT) B...",[The multilingual approaches compared with are...
3,1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfd...,best_adaptive_ir_extract_then_rule_compose,extractive,extractive,ir_rule,5,0.0,1.0,0.750000,0.0,0.400000,1.0,noisy_extraction,NMT MNMT BIBREF19 BIBREF16,"[BIBREF19, BIBREF20, multilingual NMT (MNMT) B...",[We compare our approaches with related approa...
4,1912.01214:b6f15fb6279b82e34a5bf4828b7b5ddabfd...,best_adaptive_hybrid_extract_then_rule_compose,extractive,extractive,hybrid,5,0.0,1.0,0.750000,0.0,0.400000,1.0,noisy_extraction,NMT MNMT BIBREF19 BIBREF16,"[BIBREF19, BIBREF20, multilingual NMT (MNMT) B...",[The multilingual approaches compared with are...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,1801.05147:2df4a045a9cd7b44874340b6fdf9308d3c5...,dense_pool20_cross_slm_extract_then_rule_compose,unanswerable,extractive,slm_rule,1,0.0,1.0,0.095238,0.0,0.000000,0.0,noisy_extraction,The crowdsourcing platform used is Amazon Mech...,"[unanswerable, They did not use any platform, ...",[The crowdsourcing platform used is Amazon Mec...
76,1801.05147:2df4a045a9cd7b44874340b6fdf9308d3c5...,oracle_evidence_slm_extract_then_rule_compose,unanswerable,extractive,slm_rule,1,0.0,1.0,1.000000,1.0,1.000000,0.5,noisy_extraction,Unanswerable,"[unanswerable, They did not use any platform, ...",[Unanswerable]
77,1811.00383:a313e98994fc039a82aa2447c411dda92c6...,best_adaptive_direct_router,unanswerable,extractive,direct_slm,3,0.0,1.0,0.363636,0.0,0.125000,0.5,noisy_extraction,They match words before reordering them using ...,"[unanswerable, CFILT-preorder system]",[We use the CFILT-preorder system for reorderi...
78,1811.00383:a313e98994fc039a82aa2447c411dda92c6...,best_adaptive_slm_extract_then_slm_answer,unanswerable,extractive,slm,1,0.0,1.0,0.000000,0.0,0.000000,0.0,noisy_extraction,The text does not provide specific evidence ab...,"[unanswerable, CFILT-preorder system]",[The text does not provide specific evidence a...
